### Chapter 5.6: 초음파 화질 개선 - CycleGAN
### 개원 의사의 필수 도구

### Chapter 5.6: 초음파 화질 개선 (CycleGAN)
#
### 저화질 → 고화질 초음파 변환

##### 1. 라이브러리 설치


In [ ]:
# !pip install torch torchvision pillow numpy matplotlib tqdm scikit-image

In [ ]:
import os
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

##### 2. 데이터셋 다운로드 안내


In [ ]:
print("="*70)
print("MICCAI 2023 Ultrasound Enhancement Challenge")
print("="*70)
print("\n다운로드 페이지:")
print("  https://ultrasoundenhance2023.grand-challenge.org/datasets/")
print("\n비밀번호: f0ry")
print("\n데이터 구성:")
print("  - Training Data (저화질/고화질 초음파)")
print("  - Testing Data (저화질 초음파)")
print("\n폴더 구조:")
print("  data/ultrasound/")
print("    ├── trainA/  (저화질)")
print("    ├── trainB/  (고화질)")
print("    ├── testA/   (저화질 테스트)")
print("    └── testB/   (고화질 테스트)")
print("="*70)

##### 3. 데이터 경로 설정


In [ ]:
DATA_DIR = './data/ultrasound'
TRAIN_A = os.path.join(DATA_DIR, 'trainA')  # 저화질
TRAIN_B = os.path.join(DATA_DIR, 'trainB')  # 고화질

print(f"데이터 경로: {DATA_DIR}")

# 폴더 존재 확인
if os.path.exists(TRAIN_A) and os.path.exists(TRAIN_B):
    print(f"✅ 데이터 준비 완료")
else:
    print(f"⚠️  데이터 폴더가 없습니다.")
    print("위 안내에 따라 데이터를 다운로드하세요.")

##### 4. 데이터 전처리


In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])  # [-1, 1]
])

print("✅ Transform 정의 완료")

##### 5. CycleGAN Generator (U-Net 구조)


In [ ]:
# Residual Block
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3),
            nn.InstanceNorm2d(channels)
        )

    def forward(self, x):
        return x + self.block(x)

print("✅ ResidualBlock 정의")

In [ ]:
# U-Net Generator
class UNetGenerator(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, num_residual_blocks=9):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_channels, 64, kernel_size=7),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            # Downsampling
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace=True),
        )

        # Residual blocks
        residual_blocks = []
        for _ in range(num_residual_blocks):
            residual_blocks.append(ResidualBlock(256))
        self.residual_blocks = nn.Sequential(*residual_blocks)

        # Decoder
        self.decoder = nn.Sequential(
            # Upsampling
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            # Output
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, out_channels, kernel_size=7),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.residual_blocks(x)
        x = self.decoder(x)
        return x

In [ ]:
# Generator 생성
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

G_AB = UNetGenerator().to(device)  # A(저화질) → B(고화질)
G_BA = UNetGenerator().to(device)  # B(고화질) → A(저화질)

print(f"✅ Generator 생성 완료")
print(f"  - G_AB 파라미터: {sum(p.numel() for p in G_AB.parameters()):,}")
print(f"  - G_BA 파라미터: {sum(p.numel() for p in G_BA.parameters()):,}")

##### 6. PatchGAN Discriminator


In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()

        def discriminator_block(in_filters, out_filters, normalize=True):
            layers = [nn.Conv2d(in_filters, out_filters, kernel_size=4, stride=2, padding=1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *discriminator_block(in_channels, 64, normalize=False),
            *discriminator_block(64, 128),
            *discriminator_block(128, 256),
            *discriminator_block(256, 512),
            nn.Conv2d(512, 1, kernel_size=4, padding=1)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
D_A = PatchDiscriminator().to(device)
D_B = PatchDiscriminator().to(device)

print(f"✅ Discriminator 생성 완료")
print(f"  - D_A 파라미터: {sum(p.numel() for p in D_A.parameters()):,}")
print(f"  - D_B 파라미터: {sum(p.numel() for p in D_B.parameters()):,}")

##### 7. 더미 입력 테스트


In [ ]:
# 더미 저화질 초음파
dummy_low = torch.randn(2, 1, 256, 256).to(device)

# Generator 테스트
G_AB.eval()
with torch.no_grad():
    dummy_high = G_AB(dummy_low)

print(f"✅ Generator 테스트 완료")
print(f"  - 입력 (저화질): {dummy_low.shape}")
print(f"  - 출력 (고화질): {dummy_high.shape}")

In [ ]:
# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# 입력
img_low = dummy_low[0, 0].cpu().numpy()
img_low = (img_low + 1) / 2.0  # [-1, 1] → [0, 1]

axes[0].imshow(img_low, cmap='gray')
axes[0].set_title('Input (Low Quality)')
axes[0].axis('off')

# 출력
img_high = dummy_high[0, 0].cpu().numpy()
img_high = (img_high + 1) / 2.0

axes[1].imshow(img_high, cmap='gray')
axes[1].set_title('Output (Enhanced)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

##### 8. 손실 함수 및 최적화


In [ ]:
# 손실 함수
criterion_GAN = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_identity = nn.L1Loss()

# 최적화
g_optimizer = torch.optim.Adam(
    list(G_AB.parameters()) + list(G_BA.parameters()),
    lr=0.0002,
    betas=(0.5, 0.999)
)

d_A_optimizer = torch.optim.Adam(D_A.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_B_optimizer = torch.optim.Adam(D_B.parameters(), lr=0.0002, betas=(0.5, 0.999))

# 하이퍼파라미터
LAMBDA_CYCLE = 10.0
LAMBDA_IDENTITY = 0.5

print("✅ 손실 함수 및 최적화 설정 완료")

##### 9. 학습 루프 (1 에포크 예시)


In [ ]:
# 실제 데이터가 있을 때 실행
"""
G_AB.train()
G_BA.train()
D_A.train()
D_B.train()

for batch_idx, (real_A, real_B) in enumerate(train_loader):
    real_A = real_A.to(device)
    real_B = real_B.to(device)
    batch_size = real_A.size(0)

    valid = torch.ones(batch_size, 1, 16, 16).to(device)
    fake = torch.zeros(batch_size, 1, 16, 16).to(device)

    # --- Generator ---
    g_optimizer.zero_grad()

    # Identity loss
    loss_id_A = criterion_identity(G_BA(real_A), real_A)
    loss_id_B = criterion_identity(G_AB(real_B), real_B)
    loss_identity = (loss_id_A + loss_id_B) / 2

    # GAN loss
    fake_B = G_AB(real_A)
    loss_GAN_AB = criterion_GAN(D_B(fake_B), valid)

    fake_A = G_BA(real_B)
    loss_GAN_BA = criterion_GAN(D_A(fake_A), valid)

    loss_GAN = (loss_GAN_AB + loss_GAN_BA) / 2

    # Cycle loss
    recovered_A = G_BA(fake_B)
    loss_cycle_A = criterion_cycle(recovered_A, real_A)

    recovered_B = G_AB(fake_A)
    loss_cycle_B = criterion_cycle(recovered_B, real_B)

    loss_cycle = (loss_cycle_A + loss_cycle_B) / 2

    # Total generator loss
    g_loss = loss_GAN + LAMBDA_CYCLE * loss_cycle + LAMBDA_IDENTITY * loss_identity
    g_loss.backward()
    g_optimizer.step()

    # --- Discriminator A ---
    d_A_optimizer.zero_grad()

    loss_real_A = criterion_GAN(D_A(real_A), valid)
    loss_fake_A = criterion_GAN(D_A(fake_A.detach()), fake)
    loss_D_A = (loss_real_A + loss_fake_A) / 2

    loss_D_A.backward()
    d_A_optimizer.step()

    # --- Discriminator B ---
    d_B_optimizer.zero_grad()

    loss_real_B = criterion_GAN(D_B(real_B), valid)
    loss_fake_B = criterion_GAN(D_B(fake_B.detach()), fake)
    loss_D_B = (loss_real_B + loss_fake_B) / 2

    loss_D_B.backward()
    d_B_optimizer.step()

    if (batch_idx + 1) % 10 == 0:
        print(f"Batch [{batch_idx+1}], G Loss: {g_loss.item():.4f}, "
              f"D Loss: {(loss_D_A.item() + loss_D_B.item())/2:.4f}")
"""

print("학습 루프는 실제 데이터가 있을 때 실행하세요.")

##### 10. 화질 평가 메트릭 - PSNR


In [ ]:
def calculate_psnr(img1, img2):
    """PSNR (Peak Signal-to-Noise Ratio) 계산"""
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')

    max_pixel = 1.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

print("✅ PSNR 함수 정의")

In [ ]:
# 예시 계산
img1 = np.random.rand(256, 256)
img2 = img1 + np.random.randn(256, 256) * 0.1

psnr_value = calculate_psnr(img1, img2)
print(f"예시 PSNR: {psnr_value:.2f} dB")

##### 11. 화질 평가 메트릭 - SSIM

In [ ]:
from skimage.metrics import structural_similarity as ssim

def calculate_ssim(img1, img2):
    """SSIM (Structural Similarity Index) 계산"""
    return ssim(img1, img2, data_range=1.0)

print("✅ SSIM 함수 정의")

In [ ]:
# 예시 계산
ssim_value = calculate_ssim(img1, img2)
print(f"예시 SSIM: {ssim_value:.4f}")

##### 12. 화질 개선 결과 비교


In [ ]:
# 테스트 이미지 로드 (실제 파일로 변경 필요)
TEST_IMAGE_PATH = os.path.join(DATA_DIR, 'testA', 'sample.png')

if os.path.exists(TEST_IMAGE_PATH):
    test_img = Image.open(TEST_IMAGE_PATH).convert('L')
    test_tensor = transform(test_img).unsqueeze(0).to(device)

    # 화질 개선
    G_AB.eval()
    with torch.no_grad():
        enhanced_tensor = G_AB(test_tensor)

    # NumPy 변환
    test_np = test_tensor.cpu().numpy()[0, 0]
    enhanced_np = enhanced_tensor.cpu().numpy()[0, 0]

    test_np = (test_np + 1) / 2.0
    enhanced_np = (enhanced_np + 1) / 2.0

In [ ]:
# 시각화
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    axes[0].imshow(test_np, cmap='gray')
    axes[0].set_title('저화질 초음파')
    axes[0].axis('off')

    axes[1].imshow(enhanced_np, cmap='gray')
    axes[1].set_title('AI 화질 개선')
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig('ultrasound_enhancement.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("결과 저장: ultrasound_enhancement.png")

else:
    print(f"⚠️  테스트 이미지가 없습니다: {TEST_IMAGE_PATH}")

##### 13. 모델 저장


In [ ]:
SAVE_DIR = './saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(G_AB.state_dict(), os.path.join(SAVE_DIR, 'cyclegan_AB.pth'))
torch.save(G_BA.state_dict(), os.path.join(SAVE_DIR, 'cyclegan_BA.pth'))
torch.save(D_A.state_dict(), os.path.join(SAVE_DIR, 'cyclegan_DA.pth'))
torch.save(D_B.state_dict(), os.path.join(SAVE_DIR, 'cyclegan_DB.pth'))

print(f"✅ 모델 저장 완료: {SAVE_DIR}")

##### 14. CycleGAN 개념 요약


In [ ]:
print("\n" + "="*60)
print("CycleGAN 핵심 개념")
print("="*60)

print("\n1. Unpaired 학습:")
print("  - 저화질/고화질 이미지가 1:1 매칭 불필요")
print("  - 각 도메인 이미지만 있으면 학습 가능")

print("\n2. Cycle Consistency Loss:")
print("  - A → B → A 변환 시 원본 복원")
print("  - B → A → B 변환 시 원본 복원")
print("  - 해부학적 구조 보존")

print("\n3. 의료 초음파 적용:")
print("  - 저가 장비 → 고급 장비 화질")
print("  - 실시간 화질 개선 가능")
print("  - 진단 정확도 향상")

print("="*60)